# Pandas Notebook

Pandas is the go-to library for **tabular data** in Python — think Excel, but in code.  
It builds on NumPy and gives you two key data structures:
- **Series**: a labeled 1D array (one column of a table)
- **DataFrame**: a labeled 2D table (multiple columns, each a Series)

**Our running example: An Employee HR Dataset 🏢**  
We track 10 employees across 5 cities with their Age, Salary, Experience, and Performance Score.
Every concept in this notebook uses this single consistent dataset, making it easy to compare results across sections.

In [2]:
!pip3 install pandas

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.precision", 2)
pd.set_option("display.width", 100)

## 1) Mental Model: dict of Series + axis

A **DataFrame** is best understood as a **dictionary of Series**:
- Each key is a column name
- Each value is a Series (indexed list of values)

```
df = {"Name": Series([...]), "Age": Series([...]), ...}
```

The **axis** argument tells pandas *which direction* to operate:
- `axis=0` → down the rows (default, most aggregations are per column)
- `axis=1` → across the columns

In [2]:
# 🏢 BUILD our Employee HR DataFrame
# Think of each key as a spreadsheet column, each list as the column values
df = pd.DataFrame({
    "Name":             ["Ana", "Ben", "Cara", "Dan", "Eva", "Finn", "Gina", "Hugo", "Ivy", "Jack"],
    "Age":              [28, 35, 22, 40, 31, 27, 24, 38, 29, 33],
    "City":             ["NY", "Paris", "NY", "London", "Paris", "Berlin", "NY", "London", "Berlin", "Paris"],
    "Salary":           [52000, 68000, 45000, 82000, 61000, 54000, 48000, 79000, 56000, 65000],
    "Experience_Years": [4, 10, 1, 15, 7, 3, 2, 12, 5, 9],
    "Performance_Score":[3.8, 4.2, 3.5, 4.6, 4.1, 3.9, 3.6, 4.4, 4.0, 4.3],
})
df  # display the full table

,Name,Age,City,Salary,Experience_Years,Performance_Score
0,Ana,28,NY,52000,4,3.8
1,Ben,35,Paris,68000,10,4.2
2,Cara,22,NY,45000,1,3.5
3,Dan,40,London,82000,15,4.6
4,Eva,31,Paris,61000,7,4.1
5,Finn,27,Berlin,54000,3,3.9
6,Gina,24,NY,48000,2,3.6
7,Hugo,38,London,79000,12,4.4
8,Ivy,29,Berlin,56000,5,4.0
9,Jack,33,Paris,65000,9,4.3


In [3]:
# 🏢 Column names — quick sanity check before any analysis
df.columns.tolist()

['Name', 'Age', 'City', 'Salary', 'Experience_Years', 'Performance_Score']

In [4]:
# 🏢 Selecting ONE column returns a SERIES (not a DataFrame)
# Example: get the 'Name' column — notice it has an index on the left
df["Name"]

0     Ana
1     Ben
2    Cara
3     Dan
4     Eva
5    Finn
6    Gina
7    Hugo
8     Ivy
9    Jack
Name: Name, dtype: object

In [5]:
# 🏢 Confirm the types: one selection → Series, whole table → DataFrame
type(df), type(df["Name"])
# Output: (DataFrame, Series)

(pandas.core.frame.DataFrame, pandas.core.series.Series)

In [6]:
# 🏢 axis=0 → aggregate DOWN the rows (i.e., compute one value per column)
# Average age across all employees
df["Age"].mean(axis=0)   # axis=0 means: collapse all rows → give one number per column

np.float64(30.7)

## 2) Selection (loc), Filtering, Assignment

Pandas gives you powerful ways to **select** and **filter** data:

| Method | Syntax | Use case |
|--------|--------|-----------|
| `df.loc[row, col]` | Label-based | Select by row label and column name |
| `df[condition]` | Boolean mask | Filter rows that match a condition |
| `df["col"] = ...` | Assignment | Create or update a column |

In [7]:
# 🏢 loc — label-based selection
# Get the Name of the employee at row index 0 (Ana)
df.loc[0, "Name"]

'Ana'

In [8]:
# 🏢 loc with slice — get Name and Age of first 2 employees
# Useful when HR wants a quick preview of a subset of columns
df.loc[0:1, ["Name", "Age"]]

,Name,Age
0,Ana,28
1,Ben,35


In [9]:
# 🏢 Boolean filter — find employees older than 25
# The condition df["Age"] > 25 creates a True/False mask for each row
df[df["Age"] > 25]

,Name,Age,City,Salary,Experience_Years,Performance_Score
0,Ana,28,NY,52000,4,3.8
1,Ben,35,Paris,68000,10,4.2
3,Dan,40,London,82000,15,4.6
4,Eva,31,Paris,61000,7,4.1
5,Finn,27,Berlin,54000,3,3.9
7,Hugo,38,London,79000,12,4.4
8,Ivy,29,Berlin,56000,5,4.0
9,Jack,33,Paris,65000,9,4.3


In [10]:
# 🏢 Multi-condition filter: NY office employees under 30
# Use & (AND) or | (OR) — always wrap each condition in parentheses!
df[(df["City"] == "NY") & (df["Age"] < 30)]

,Name,Age,City,Salary,Experience_Years,Performance_Score
0,Ana,28,NY,52000,4,3.8
2,Cara,22,NY,45000,1,3.5
6,Gina,24,NY,48000,2,3.6


In [11]:
# 🏢 ASSIGNMENT — create a new column 'Is_Senior' based on a condition
# Employees with >= 10 years of experience are considered senior
df["Is_Senior"] = df["Experience_Years"] >= 10
df[["Name", "Experience_Years", "Is_Senior"]]  # show only relevant columns

,Name,Experience_Years,Is_Senior
0,Ana,4,False
1,Ben,10,True
2,Cara,1,False
3,Dan,15,True
4,Eva,7,False
5,Finn,3,False
6,Gina,2,False
7,Hugo,12,True
8,Ivy,5,False
9,Jack,9,False


## 3) Inspection

Before you analyze any dataset, **always inspect it first.**  
These methods answer: *What does my data look like? How big is it? Any surprises?*

| Method | What it tells you |
|--------|-------------------|
| `df.head(n)` | First n rows |
| `df.tail(n)` | Last n rows |
| `df.info()` | Column types and non-null counts |
| `df.describe()` | Stats (mean, std, min, max, quartiles) |
| `df.shape` | (rows, columns) |
| `df.columns` | List of column names |

In [12]:
# 🏢 head() — quickly peek at the top of the dataset
# Default is 5 rows; handy after loading a large CSV
df.head()

,Name,Age,City,Salary,Experience_Years,Performance_Score,Is_Senior
0,Ana,28,NY,52000,4,3.8,False
1,Ben,35,Paris,68000,10,4.2,True
2,Cara,22,NY,45000,1,3.5,False
3,Dan,40,London,82000,15,4.6,True
4,Eva,31,Paris,61000,7,4.1,False


In [13]:
# 🏢 info() — the 'X-ray' of your DataFrame
# Shows column names, non-null counts, and dtypes
# Use this FIRST after loading new data — it reveals missing values and wrong types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Name               10 non-null     object 
 1   Age                10 non-null     int64  
 2   City               10 non-null     object 
 3   Salary             10 non-null     int64  
 4   Experience_Years   10 non-null     int64  
 5   Performance_Score  10 non-null     float64
 6   Is_Senior          10 non-null     bool   
dtypes: bool(1), float64(1), int64(3), object(2)
memory usage: 622.0+ bytes


In [14]:
# 🏢 describe() — statistical summary of numeric columns
# include='all' also includes string/object columns (shows unique counts, top value)
df.describe(include="all")

,Name,Age,City,Salary,Experience_Years,Performance_Score,Is_Senior
count,10,10.00,10,10.00,10.00,10.00,10
unique,10,NaN,4,NaN,NaN,NaN,2
top,Ana,NaN,NY,NaN,NaN,NaN,False
freq,1,NaN,3,NaN,NaN,NaN,7
mean,NaN,30.70,NaN,61000.00,6.80,4.04,NaN
std,NaN,5.85,NaN,12516.66,4.61,0.35,NaN
min,NaN,22.00,NaN,45000.00,1.00,3.50,NaN
25%,NaN,27.25,NaN,52500.00,3.25,3.82,NaN
50%,NaN,30.00,NaN,58500.00,6.00,4.05,NaN
75%,NaN,34.50,NaN,67250.00,9.75,4.28,NaN


In [15]:
# 🏢 shape and columns — dimensions & column names
print("Shape (rows, cols):", df.shape)
print("Columns:", df.columns.tolist())

Shape (rows, cols): (10, 7)
Columns: ['Name', 'Age', 'City', 'Salary', 'Experience_Years', 'Performance_Score', 'Is_Senior']


## 4) Cleaning

Real-world data is messy! Common issues:
- **Missing values (NaN)** — someone forgot to fill in a field
- **Duplicate rows** — the same record entered twice

Pandas tools:
| Method | What it does |
|--------|--------------|
| `df.isna()` | Boolean mask: True where values are missing |
| `df.dropna()` | Remove rows (or cols) with any missing value |
| `df.fillna(val)` | Replace NaN with a given value |
| `df.drop_duplicates()` | Keep only unique rows |

In [16]:
# 🏢 SIMULATE a messy HR extract — missing salary data (employee didn't disclose)
messy = pd.DataFrame({
    "Name":   ["Zara", "Omar", "Priya"],
    "Salary": [72000.0, np.nan, 68000.0],   # Omar's salary is unknown
    "City":   ["NY",    "NY",   "NY"]
})
messy

,Name,Salary,City
0,Zara,72000.0,NY
1,Omar,NaN,NY
2,Priya,68000.0,NY


In [17]:
# 🏢 dropna() — remove rows where ANY value is missing
# Omar's row is removed because his Salary is NaN
messy.dropna()

,Name,Salary,City
0,Zara,72000.0,NY
2,Priya,68000.0,NY


In [18]:
# 🏢 fillna() — fill missing salary with the median salary (a safe default)
# This is better than dropping rows when you want to preserve all records
median_salary = messy["Salary"].median()
filled = messy.fillna({"Salary": median_salary})
print(f"Median salary used for fill: {median_salary}")
filled

Median salary used for fill: 70000.0


,Name,Salary,City
0,Zara,72000.0,NY
1,Omar,70000.0,NY
2,Priya,68000.0,NY


In [19]:
# 🏢 DROP DUPLICATES — the same employee record was submitted twice
with_dupes = pd.concat([messy, messy.iloc[[0]]], ignore_index=True)
print("With duplicate:\n", with_dupes)
print("\nAfter drop_duplicates():\n", with_dupes.drop_duplicates())

With duplicate:
     Name   Salary City
0   Zara  72000.0   NY
1   Omar      NaN   NY
2  Priya  68000.0   NY
3   Zara  72000.0   NY

After drop_duplicates():
     Name   Salary City
0   Zara  72000.0   NY
1   Omar      NaN   NY
2  Priya  68000.0   NY


## 5) GroupBy (split-apply-combine)

**GroupBy** is how pandas answers questions like:  
*"What is the average salary per city?"* or *"Who is the top performer in each department?"*

The pattern is always:
1. **Split** — divide rows into groups based on a column value
2. **Apply** — run an aggregation (mean, sum, max, count, …)
3. **Combine** — collect results into a new Series or DataFrame

```python
df.groupby("City")["Salary"].mean()
#          ^^^^^^  ^^^^^^^^  ^^^^^^
#          group   column    agg
```

In [20]:
# 🏢 Average salary per city — which office pays more?
df.groupby("City")["Salary"].mean().sort_values(ascending=False)

City
London    80500.00
Paris     64666.67
Berlin    55000.00
NY        48333.33
Name: Salary, dtype: float64

In [21]:
# 🏢 Multiple aggregations at once using .agg()
# Find: headcount, average salary, and max performance score per city
df.groupby("City").agg(
    Headcount       = ("Name",              "count"),
    Avg_Salary      = ("Salary",            "mean"),
    Top_Performance = ("Performance_Score", "max")
)

,Headcount,Avg_Salary,Top_Performance
City,,,
Berlin,2,55000.00,4.0
London,2,80500.00,4.6
NY,3,48333.33,3.8
Paris,3,64666.67,4.3


In [22]:
# 🏢 Group by 'Is_Senior' — compare senior vs junior employee stats
df.groupby("Is_Senior")[["Salary", "Performance_Score", "Age"]].mean()

,Salary,Performance_Score,Age
Is_Senior,,,
False,54428.57,3.89,27.71
True,76333.33,4.40,37.67


## 6) Concatenation and Merge

Very often your data comes from **multiple sources** and needs to be combined.

| Function | Analogy | When to use |
|----------|---------|-------------|
| `pd.concat([df1, df2])` | Stacking pages | Same columns, more rows (e.g. monthly data batches) |
| `pd.merge(left, right, on=key, how=type)` | SQL JOIN | Different tables sharing a common key |

**Merge types:**
- `inner` — only rows where the key exists in BOTH tables
- `left` — all rows from the left table, NaN where right has no match
- `right` — all rows from the right table, NaN where left has no match
- `outer` — all rows from both tables, NaN where no match

In [23]:
# 🏢 CONCAT — combine January and February new hires into one DataFrame
jan_hires = pd.DataFrame({
    "Name": ["Ravi", "Sara"],
    "City": ["Mumbai", "Delhi"],
    "Salary": [55000, 62000]
})
feb_hires = pd.DataFrame({
    "Name": ["Tom", "Lily"],
    "City": ["NY", "London"],
    "Salary": [70000, 68000]
})
all_hires = pd.concat([jan_hires, feb_hires], ignore_index=True)
print("All new hires combined:")
all_hires

All new hires combined:


,Name,City,Salary
0,Ravi,Mumbai,55000
1,Sara,Delhi,62000
2,Tom,NY,70000
3,Lily,London,68000


In [24]:
# 🏢 MERGE (inner join) — employee table + bonus table, matching on employee Name
# Only employees who appear in BOTH tables are returned
emp_info   = pd.DataFrame({"Name": ["Ana", "Ben",  "Cara"], "City": ["NY", "Paris", "NY"]})
bonus_info = pd.DataFrame({"Name": ["Ben", "Cara", "Dan"],  "Bonus": [5000, 3000, 8000]})

print("Inner join — only matched employees:")
pd.merge(emp_info, bonus_info, on="Name", how="inner")

Inner join — only matched employees:


,Name,City,Bonus
0,Ben,Paris,5000
1,Cara,NY,3000


In [25]:
# 🏢 MERGE (left join) — keep ALL employees, fill NaN for those with no bonus record
print("Left join — all employees shown, NaN where bonus missing:")
pd.merge(emp_info, bonus_info, on="Name", how="left")

Left join — all employees shown, NaN where bonus missing:


,Name,City,Bonus
0,Ana,NY,NaN
1,Ben,Paris,5000.0
2,Cara,NY,3000.0


## 7) Map and Apply

Use these to **transform values element-by-element** or **row-by-row**:

| Method | Operates on | Use case |
|--------|-------------|----------|
| `Series.map(dict)` | Each element | Lookup-table replacement (e.g. encode gender) |
| `Series.map(func)` | Each element | Apply a function element-wise |
| `df.apply(func, axis=1)` | Each row | Combine multiple columns into a new value |

In [26]:
# 🏢 MAP with a dictionary — encode City to a region code
# This is useful before feeding data to a machine learning model (which needs numbers)
region_map = {"NY": "NA", "Paris": "EU", "London": "EU", "Berlin": "EU"}
df["Region"] = df["City"].map(region_map)
df[["Name", "City", "Region"]].head(7)

,Name,City,Region
0,Ana,NY,NA
1,Ben,Paris,EU
2,Cara,NY,NA
3,Dan,London,EU
4,Eva,Paris,EU
5,Finn,Berlin,EU
6,Gina,NY,NA


In [27]:
# 🏢 MAP with a function — get initials from Name
df["Initial"] = df["Name"].map(lambda name: name[0] + ".")
df[["Name", "Initial"]].head(5)

,Name,Initial
0,Ana,A.
1,Ben,B.
2,Cara,C.
3,Dan,D.
4,Eva,E.


In [28]:
# 🏢 APPLY (axis=1) — create a per-row summary combining multiple columns
# Generates a human-readable 'Profile' string for each employee
def make_profile(row):
    tier = "Senior" if row["Is_Senior"] else "Junior"
    return f"{row['Name']} | {row['City']} | {tier} | Score: {row['Performance_Score']}"

df["Profile"] = df.apply(make_profile, axis=1)  # axis=1 → call function once per ROW
df["Profile"]

0         Ana | NY | Junior | Score: 3.8
1      Ben | Paris | Senior | Score: 4.2
2        Cara | NY | Junior | Score: 3.5
3     Dan | London | Senior | Score: 4.6
4      Eva | Paris | Junior | Score: 4.1
5    Finn | Berlin | Junior | Score: 3.9
6        Gina | NY | Junior | Score: 3.6
7    Hugo | London | Senior | Score: 4.4
8     Ivy | Berlin | Junior | Score: 4.0
9     Jack | Paris | Junior | Score: 4.3
Name: Profile, dtype: object